In [ ]:
# correlation between interest rates and the housing market
import os

import matplotlib.pyplot as plt
import pandas as pd
import yfinance as yf
from fredapi import Fred

yf.pdr_override()

plt.ion()

fred = Fred(api_key=os.environ["FRED_API_KEY"])

In [ ]:
def plot_it(data: pd.Series, title: str, figsize: tuple[int, int] = (15, 5)) -> None:
    fig, ax = plt.subplots(figsize=figsize)
    ax.plot(data)
    ax.grid(True, which="both")
    plt.title(title)

In [ ]:
irate = fred.get_series("FEDFUNDS").dropna()
df_home_prices = fred.get_series("MSPUS").dropna()

fig, ax = plt.subplots(figsize=(15, 5))
irate.plot(
    ax=ax,
    secondary_y=True,
    color="red",
    label="Federal Funds Rate",
)
df_home_prices.plot(ax=ax, color="blue", label="Median Home Sales Price")

plt.title("Federal Funds Rate vs Median Home Sales Price")
plt.legend(["Federal Funds Rate", "Median Home Sales Price"])
plt.show()

In [ ]:
# Correlation Coefficient = 0.6: A moderate positive relationship.
print(
    f"Federal Funds Rate correlation with Median Home Sales Price: {irate.corr(df_home_prices, method='pearson')}"
)

In [ ]:
# Get the data for the S&P 500 from Yahoo Finance
sp5 = yf.Ticker("^GSPC").history(start="1980-01-01", interval="1mo")
sp5.head()

In [ ]:
# Change the date format to YYYY-MM-DD
sp5.index = sp5.index.to_series().apply(
    lambda x: pd.Timestamp(pd.Timestamp(x).strftime("%Y-%m-%d"))
)
sp5

In [ ]:
# Make sure that both dataframes start and end at the same index
start_date = max(sp5.index.min(), irate.index.min())
end_date = min(sp5.index.max(), irate.index.max())

print(f"Start Date: {start_date}, End Date: {end_date}")

sp5 = sp5.loc[start_date:end_date]
irate = irate.loc[start_date:end_date]

In [ ]:
sp5

In [ ]:
sp5 = sp5["Close"]
roll_corr = sp5.rolling(24).corr(irate)


fig, ax = plt.subplots(figsize=(15, 5))

roll_corr.plot(
    ax=ax,
    color="blue",
    label="Rolling Correlation",
    secondary_y=True,
    linestyle="--",
)
irate.plot(
    ax=ax,
    color="red",
    label="Federal Funds Rate",
    secondary_y=True,
)
sp5.plot(ax=ax, color="green", label="S&P500", secondary_y=False)


plt.legend(["Rolling Correlation", "Federal Funds Rate", "S&P500"])
plt.title("Rolling Correlation between Federal Funds Rate and S&P500")
plt.show()

In [ ]:
# Spikes in the cost of borrowing, and in energy prices, have been forerunners of recession in the past

# Market Yield on U.S. Treasury Securities at 10-Year Constant Maturity, Quoted on an Investment Basis (DGS10)
df_treasury = fred.get_series("DGS10").dropna()

# Crude Oil Prices: West Texas Intermediate (WTI) - Cushing, Oklahoma (DCOILWTICO)
df_oil = fred.get_series("DCOILWTICO").dropna()
print(
    f"Correlation between treasury yields and crude oil prices: {df_treasury.corr(df_oil, method='pearson')}"
)

In [ ]:
print(f"Correlation of S&P500 and Federal Interest Rates: {sp5.corr(irate)}")

In [ ]:
# RELATIONSHIP BETWEEN INTEREST RATES AND INFLATION
# Sticky Price Consumer Price Index less Food and Energy (CORESTICKM159SFRBATL)
data = fred.get_series("CORESTICKM159SFRBATL")
inflation = data.dropna()

fig, ax = plt.subplots(figsize=(15, 5))
inflation.plot(ax=ax, color="blue", label="Inflation", secondary_y=True)
irate.plot(ax=ax, color="red", label="Federal Funds Rate")
plt.title("Inflation vs Federal Funds Rate")
plt.legend(["Inflation", "Federal Funds Rate"])
plt.show()

In [ ]:
irate.corr(inflation, method="pearson")

In [ ]:
# NOTE: .corr() sanity check
def gdp_rate_of_change():
    gdp = fred.get_series("GDP")
    rolling_avg = gdp.rolling(window=5).mean()
    rate_of_change = rolling_avg.pct_change()
    return gdp.dropna(), rate_of_change.dropna()


ONE_YEAR = -5
gdp, gdp_roc = gdp_rate_of_change()
gdp, gdp_roc = gdp[ONE_YEAR:], gdp_roc[ONE_YEAR:]

gdp.corr(gdp_roc)

In [ ]:
# 1. Change correlation between sp500 and the federal funds rate to be a rolling correlation (look at when it was high vs low)
# 2. What was the rate of change with the SP500 when the Feds raised/lowered interest rates?
sp5_roc = sp5.diff()
sp5_roc_rolling = sp5_roc.rolling(12).mean()
sp5_roc_rolling.dropna(inplace=True)

In [ ]:
fig, ax = plt.subplots(figsize=(15, 5))
sp5_roc_rolling.plot(
    ax=ax, color="green", label="S&P500 Rate of Change", secondary_y=True
)
irate.plot(
    ax=ax,
    color="red",
    label="Federal Funds Rate",
)

plt.grid(True, which="both", axis="both")
plt.legend(["Federal Funds Rate", "S&P500 Rate of Change"])
plt.title("Federal Funds Rate and S&P500 Rate of Change")
plt.show()